In [ ]:
# ============================================================
# 05_final_biomarker_trajectories
# Thesis final pipeline
# ============================================================

import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.stats import mannwhitneyu
from statsmodels.nonparametric.smoothers_lowess import lowess
from statsmodels.stats.multitest import multipletests

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)

PROJECT_CONSTANTS_DIR = Path(
    "/home/azureuser/cloudfiles/code/Users/m.bolognini/"
    "UseCase-Code/data-science/projects/a_MB_thesis_final/"
    "pipeline_thesis_final"
)

sys.path.append(str(PROJECT_CONSTANTS_DIR))

from project_constants import *

print("OUTPUT_03:", OUTPUT_03)
print("OUTPUT_05:", OUTPUT_05)

In [ ]:
# ============================================================
# LOAD DATASETS
# ============================================================

df_model = pd.read_parquet(
    OUTPUT_03 / "03_model_features.parquet"
)

df_daily = pd.read_parquet(
    OUTPUT_03 / "03_daily_labs.parquet"
)

print("Model dataset:", df_model.shape)
print("Daily labs:", df_daily.shape)

print("Patients:", df_model["pubID"].nunique())
print("Episodes:", df_model["episode_key"].nunique())

In [ ]:
# ============================================================
# BUILD TRAJECTORY DATASET
# ============================================================

merge_cols = [
    "episode_key",
    "pubID",
    "temporal_cohort",
    "hospitalWard",
    "metastatic_group",
    "composite_inhospital_or_palliative",
    "mortality_90d_post_discharge",
]

merge_cols = [
    c for c in merge_cols
    if c in df_model.columns
]

df_traj = df_daily.merge(
    df_model[merge_cols],
    on="episode_key",
    how="left"
)

df_traj = df_traj[
    df_traj["biomarker"].isin(CORE_BIOMARKERS)
].copy()

df_traj["lab_day"] = pd.to_numeric(
    df_traj["lab_day"],
    errors="coerce"
)

df_traj["value"] = pd.to_numeric(
    df_traj["value"],
    errors="coerce"
)

df_traj = df_traj.dropna(
    subset=["episode_key", "lab_day", "value", "biomarker"]
).copy()

df_traj = df_traj[
    df_traj["lab_day"].between(0, 7)
].copy()

print("Trajectory dataset:", df_traj.shape)
print("Episodes:", df_traj["episode_key"].nunique())
print("Patients:", df_traj["pubID"].nunique())
print("Biomarkers:", df_traj["biomarker"].nunique())

print(sorted(df_traj["biomarker"].unique()))

In [ ]:
# ============================================================
# INTERPRETATION NOTE: UNADJUSTED TRAJECTORY COMPARISONS
# ============================================================

print("""
Trajectory plots and statistical comparisons stratified by outcome are descriptive
and unadjusted.

They should be interpreted as unadjusted associations, not as independent causal
or prognostic effects of individual biomarkers. Patients with adverse outcomes may
also differ in age, frailty, metastatic status, comorbidity burden, and monitoring
intensity.

Covariate-adjusted prediction analyses are performed in the supervised modelling
and Cox survival notebooks.
""")

In [ ]:
# ============================================================
# NORMALIZED VALUES: WITHIN-BIOMARKER Z-SCORE
# ============================================================

df_traj["value_zscore"] = (
    df_traj
    .groupby("biomarker")["value"]
    .transform(
        lambda x: (x - x.mean()) / x.std()
    )
)

display(df_traj.head())

In [ ]:
# ============================================================
# AVAILABILITY BY BIOMARKER AND DAY
# ============================================================

availability = (
    df_traj
    .groupby(["biomarker", "lab_day"])["episode_key"]
    .nunique()
    .reset_index(name="n_episodes")
)

availability_pivot = (
    availability
    .pivot(
        index="biomarker",
        columns="lab_day",
        values="n_episodes"
    )
    .fillna(0)
)

availability_pivot = availability_pivot.loc[
    [b for b in CORE_BIOMARKERS if b in availability_pivot.index]
]

display(availability_pivot)

In [ ]:
# ============================================================
# AVAILABILITY HEATMAP
# ============================================================

plt.figure(figsize=(12, 6))

sns.heatmap(
    availability_pivot,
    annot=True,
    fmt=".0f",
    cmap="Blues"
)

plt.title("Number of contributing episodes by biomarker and hospitalization day")
plt.xlabel("Day from admission")
plt.ylabel("Biomarker")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# DAILY MEDIAN TRAJECTORY TABLE
# ============================================================

daily_summary = (
    df_traj
    .groupby(["biomarker", "lab_day"])
    .agg(
        n_episodes=("episode_key", "nunique"),
        median_value=("value", "median"),
        q1=("value", lambda x: x.quantile(0.25)),
        q3=("value", lambda x: x.quantile(0.75)),
        median_zscore=("value_zscore", "median"),
    )
    .reset_index()
)

display(daily_summary.head(30))

In [ ]:
# ============================================================
# RAW TRAJECTORY PLOT FUNCTION
# ============================================================

def plot_biomarker_trajectories(
    data,
    strat_col,
    strat_labels=None,
    title_suffix="",
    output_path=None,
    value_col="value",
    ylabel="Daily median",
):
    biomarkers = [
        b for b in CORE_BIOMARKERS
        if b in data["biomarker"].unique()
    ]

    fig, axes = plt.subplots(
        nrows=5,
        ncols=2,
        figsize=(16, 22),
        sharex=True
    )

    axes = axes.flatten()

    for i, biomarker in enumerate(biomarkers):

        ax = axes[i]

        temp_bio = data[
            data["biomarker"] == biomarker
        ].copy()

        groups = sorted(
            temp_bio[strat_col]
            .dropna()
            .unique()
        )

        for group in groups:

            temp_group = temp_bio[
                temp_bio[strat_col] == group
            ].copy()

            summary = (
                temp_group
                .groupby("lab_day")
                .agg(
                    median_value=(value_col, "median"),
                    n_episodes=("episode_key", "nunique")
                )
                .reset_index()
                .sort_values("lab_day")
            )

            if summary.empty:
                continue

            label = (
                strat_labels.get(group, str(group))
                if strat_labels
                else str(group)
            )

            ax.plot(
                summary["lab_day"],
                summary["median_value"],
                marker="o",
                label=label
            )

            if summary.shape[0] >= 3:

                smoothed = lowess(
                    endog=summary["median_value"],
                    exog=summary["lab_day"],
                    frac=0.6,
                    return_sorted=True
                )

                ax.plot(
                    smoothed[:, 0],
                    smoothed[:, 1],
                    linestyle="--",
                    alpha=0.8
                )

        ax.set_title(biomarker)
        ax.set_xlabel("Day from admission")
        ax.set_ylabel(ylabel)
        ax.legend()

    for j in range(len(biomarkers), len(axes)):
        axes[j].axis("off")

    plt.suptitle(
        f"Daily biomarker trajectories {title_suffix}",
        fontsize=16,
        y=1.01
    )

    plt.tight_layout()

    if output_path is not None:
        plt.savefig(output_path, dpi=300, bbox_inches="tight")

    plt.show()

In [ ]:
# ============================================================
# INTERPRETATION NOTE
# ============================================================

print("""
Interpretation note:
LOWESS smoothing is used for visual interpretation only.

Later hospitalization days include progressively fewer contributing episodes
because of discharge, death, and heterogeneous monitoring intensity.

Therefore, small late-window oscillations should not be interpreted as
definitive biological reversals.
""")

In [ ]:
# ============================================================
# RAW TRAJECTORIES BY COMPOSITE OUTCOME
# ============================================================

df_composite = df_traj[
    df_traj["composite_inhospital_or_palliative"].notna()
].copy()

plot_biomarker_trajectories(
    data=df_composite,
    strat_col="composite_inhospital_or_palliative",
    strat_labels={
        0: "No composite outcome",
        1: "Composite outcome"
    },
    title_suffix="by composite outcome",
    output_path=OUTPUT_05 / "05_raw_trajectories_composite_outcome.png",
    value_col="value",
    ylabel="Daily median"
)

In [ ]:
# ============================================================
# RAW TRAJECTORIES BY 90-DAY MORTALITY
# ============================================================

df_90d = df_traj[
    df_traj["mortality_90d_post_discharge"].notna()
].copy()

plot_biomarker_trajectories(
    data=df_90d,
    strat_col="mortality_90d_post_discharge",
    strat_labels={
        0: "Alive at 90 days",
        1: "Dead within 90 days"
    },
    title_suffix="by 90-day post-discharge mortality",
    output_path=OUTPUT_05 / "05_raw_trajectories_90d_mortality.png",
    value_col="value",
    ylabel="Daily median"
)

In [ ]:
# ============================================================
# RAW TRAJECTORIES BY METASTATIC STATUS
# ============================================================

df_met = df_traj[
    df_traj["metastatic_group"].isin(
        ["Non-metastatic", "Metastatic"]
    )
].copy()

plot_biomarker_trajectories(
    data=df_met,
    strat_col="metastatic_group",
    strat_labels={
        "Non-metastatic": "Non-metastatic",
        "Metastatic": "Metastatic"
    },
    title_suffix="by metastatic status",
    output_path=OUTPUT_05 / "05_raw_trajectories_metastatic_status.png",
    value_col="value",
    ylabel="Daily median"
)

In [ ]:
# ============================================================
# NORMALIZED TRAJECTORIES BY COMPOSITE OUTCOME
# ============================================================

plot_biomarker_trajectories(
    data=df_composite,
    strat_col="composite_inhospital_or_palliative",
    strat_labels={
        0: "No composite outcome",
        1: "Composite outcome"
    },
    title_suffix="(z-score normalized) by composite outcome",
    output_path=OUTPUT_05 / "05_normalized_trajectories_composite_outcome.png",
    value_col="value_zscore",
    ylabel="Daily median z-score"
)

In [ ]:
# ============================================================
# NORMALIZED TRAJECTORIES BY 90-DAY MORTALITY
# ============================================================

plot_biomarker_trajectories(
    data=df_90d,
    strat_col="mortality_90d_post_discharge",
    strat_labels={
        0: "Alive at 90 days",
        1: "Dead within 90 days"
    },
    title_suffix="(z-score normalized) by 90-day mortality",
    output_path=OUTPUT_05 / "05_normalized_trajectories_90d_mortality.png",
    value_col="value_zscore",
    ylabel="Daily median z-score"
)

In [ ]:
# ============================================================
# NORMALIZED TRAJECTORIES BY METASTATIC STATUS
# ============================================================

plot_biomarker_trajectories(
    data=df_met,
    strat_col="metastatic_group",
    strat_labels={
        "Non-metastatic": "Non-metastatic",
        "Metastatic": "Metastatic"
    },
    title_suffix="(z-score normalized) by metastatic status",
    output_path=OUTPUT_05 / "05_normalized_trajectories_metastatic_status.png",
    value_col="value_zscore",
    ylabel="Daily median z-score"
)

In [ ]:
# ============================================================
# WINDOW COMPARISON DATASET
# ============================================================

window_rows = []

for episode_key, ep_data in df_traj.groupby("episode_key"):

    episode_info = ep_data.iloc[0]

    for biomarker in CORE_BIOMARKERS:

        temp = ep_data[
            ep_data["biomarker"] == biomarker
        ].copy()

        if temp.empty:
            continue

        row = {
            "episode_key": episode_key,
            "pubID": episode_info["pubID"],
            "biomarker": biomarker,
            "composite_inhospital_or_palliative":
                episode_info.get("composite_inhospital_or_palliative", np.nan),
            "mortality_90d_post_discharge":
                episode_info.get("mortality_90d_post_discharge", np.nan),
            "metastatic_group":
                episode_info.get("metastatic_group", np.nan),
        }

        for window_name, (start_day, end_day) in {
            "0_3": LAB_WINDOWS["0_3"],
            "3_7": LAB_WINDOWS["3_7"],
        }.items():

            w = temp[
                temp["lab_day"].between(start_day, end_day)
            ].copy()

            row[f"mean_{window_name}"] = w["value"].mean()
            row[f"median_{window_name}"] = w["value"].median()
            row[f"n_days_{window_name}"] = w["lab_day"].nunique()
        
# NOTE:
# This delta is an inter-window difference:
# mean value during days 3–7 minus mean value during days 0–3.
#
# It is different from the feature-engineering deltas in Notebook 03, where
# delta_0_3, delta_3_7, and delta_0_7 are computed as last observed value minus
# first observed value within the same temporal window.

        row["delta_3_7_vs_0_3"] = (
            row["mean_3_7"] - row["mean_0_3"]
            if pd.notna(row["mean_3_7"]) and pd.notna(row["mean_0_3"])
            else np.nan
        )

        row["delta_median_3_7_vs_0_3"] = (
            row["median_3_7"] - row["median_0_3"]
            if pd.notna(row["median_3_7"]) and pd.notna(row["median_0_3"])
            else np.nan
        )

        window_rows.append(row)

df_window_compare = pd.DataFrame(window_rows)

print("Window comparison dataset:", df_window_compare.shape)
print("Episodes:", df_window_compare["episode_key"].nunique())
print("Biomarkers:", df_window_compare["biomarker"].nunique())

display(df_window_compare.head())

In [ ]:
# ============================================================
# EFFECT SIZE FUNCTIONS
# ============================================================

def cliffs_delta(x, y):

    x = pd.Series(x).dropna().to_numpy()
    y = pd.Series(y).dropna().to_numpy()

    if len(x) == 0 or len(y) == 0:
        return np.nan

    gt = 0
    lt = 0

    for xi in x:
        gt += np.sum(xi > y)
        lt += np.sum(xi < y)

    return (gt - lt) / (len(x) * len(y))


def interpret_cliffs_delta(delta):

    if pd.isna(delta):
        return np.nan

    ad = abs(delta)

    if ad < 0.147:
        return "negligible"
    elif ad < 0.33:
        return "small"
    elif ad < 0.474:
        return "medium"
    else:
        return "large"

In [ ]:
# ============================================================
# WINDOW STATISTICAL COMPARISON FUNCTION
# ============================================================

def compare_window_values(df, outcome_col, outcome_label):

    rows = []

    for biomarker in CORE_BIOMARKERS:

        temp_bio = df[
            df["biomarker"] == biomarker
        ].copy()

        for feature in [
            "mean_0_3",
            "mean_3_7",
            "delta_3_7_vs_0_3",
        ]:

            temp = temp_bio[
                temp_bio[outcome_col].notna()
                & temp_bio[feature].notna()
            ].copy()

            if temp.empty:
                continue

            groups = sorted(
                temp[outcome_col]
                .dropna()
                .unique()
            )

            if len(groups) != 2:
                continue

            g0 = temp.loc[
                temp[outcome_col] == groups[0],
                feature
            ]

            g1 = temp.loc[
                temp[outcome_col] == groups[1],
                feature
            ]

            if len(g0) < 3 or len(g1) < 3:

                p = np.nan
                effect_size = np.nan
                effect_interpretation = np.nan

            else:

                try:
                    _, p = mannwhitneyu(
                        g0,
                        g1,
                        alternative="two-sided"
                    )

                    effect_size = cliffs_delta(g0, g1)
                    effect_interpretation = interpret_cliffs_delta(effect_size)

                except Exception:

                    p = np.nan
                    effect_size = np.nan
                    effect_interpretation = np.nan

            rows.append({
                "comparison": outcome_label,
                "outcome_col": outcome_col,
                "biomarker": biomarker,
                "feature": feature,
                "group_0": groups[0],
                "group_1": groups[1],
                "group_0_n": len(g0),
                "group_1_n": len(g1),
                "group_0_median": g0.median(),
                "group_1_median": g1.median(),
                "group_0_q1": g0.quantile(0.25),
                "group_0_q3": g0.quantile(0.75),
                "group_1_q1": g1.quantile(0.25),
                "group_1_q3": g1.quantile(0.75),
                "median_difference_group1_minus_group0": (
                    g1.median() - g0.median()
                ),
                "p_value": p,
                "effect_size_cliffs_delta": effect_size,
                "effect_size_magnitude": effect_interpretation,
            })

    return pd.DataFrame(rows)

In [ ]:
# ============================================================
# WINDOW COMPARISONS BY COMPOSITE OUTCOME
# ============================================================

window_comparison_composite = compare_window_values(
    df=df_window_compare,
    outcome_col="composite_inhospital_or_palliative",
    outcome_label="Composite outcome"
)

display(
    window_comparison_composite
    .sort_values("p_value")
    .head(30)
)

In [ ]:
# ============================================================
# WINDOW COMPARISONS BY 90-DAY MORTALITY
# ============================================================

window_comparison_90d = compare_window_values(
    df=df_window_compare,
    outcome_col="mortality_90d_post_discharge",
    outcome_label="90-day post-discharge mortality"
)

display(
    window_comparison_90d
    .sort_values("p_value")
    .head(30)
)

In [ ]:
# ============================================================
# WINDOW COMPARISONS BY METASTATIC STATUS
# ============================================================

df_window_met = df_window_compare[
    df_window_compare["metastatic_group"].isin(
        ["Non-metastatic", "Metastatic"]
    )
].copy()

window_comparison_metastatic = compare_window_values(
    df=df_window_met,
    outcome_col="metastatic_group",
    outcome_label="Metastatic vs non-metastatic"
)

display(
    window_comparison_metastatic
    .sort_values("p_value")
    .head(30)
)

In [ ]:
# ============================================================
# COMBINE WINDOW COMPARISONS WITH EFFECT SIZES
# ============================================================

window_comparisons_all = pd.concat(
    [
        window_comparison_composite,
        window_comparison_90d,
        window_comparison_metastatic,
    ],
    ignore_index=True
)

display(
    window_comparisons_all
    .sort_values("p_value")
    .head(50)
)

print("Total comparisons:", window_comparisons_all.shape[0])
print(
    "Nominally significant p < 0.05:",
    window_comparisons_all["p_value"].lt(0.05).sum()
)

In [ ]:
# ============================================================
# EFFECT SIZE SUMMARY
# ============================================================

effect_size_summary = (
    window_comparisons_all
    .groupby(
        [
            "comparison",
            "effect_size_magnitude"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="n_comparisons")
)

display(effect_size_summary)

display(
    window_comparisons_all
    .sort_values(
        "effect_size_cliffs_delta",
        key=lambda x: x.abs(),
        ascending=False
    )
    .head(30)
)

In [ ]:
# ============================================================
# MULTIPLE TESTING CORRECTION
# Benjamini-Hochberg FDR correction
# ============================================================

window_comparisons_all["p_value_fdr"] = np.nan
window_comparisons_all["significant_fdr"] = False

valid_p = window_comparisons_all["p_value"].notna()

reject, p_fdr, _, _ = multipletests(
    window_comparisons_all.loc[valid_p, "p_value"],
    method="fdr_bh"
)

window_comparisons_all.loc[valid_p, "p_value_fdr"] = p_fdr
window_comparisons_all.loc[valid_p, "significant_fdr"] = reject

print("Nominally significant p < 0.05:")
print(window_comparisons_all["p_value"].lt(0.05).sum())

print("\nSignificant after FDR correction:")
print(window_comparisons_all["significant_fdr"].sum())

display(
    window_comparisons_all
    .sort_values("p_value_fdr")
    .head(30)
)

In [ ]:
# ============================================================
# LIGHT TRAJECTORY DIRECTION CLASSIFICATION
# Biomarker-specific clinically interpretable thresholds
# ============================================================

trajectory_delta_thresholds = {
    # inflammatory markers
    "CRP": 20.0,          # mg/L , decidere se 20.0 o 30.0 (più conservativo)
    "WBC": 1.0,           # 10^9/L
    "Neutrophils": 1.0,   # 10^9/L
    "Lymphocytes": 0.2,   # 10^9/L

    # hematology
    "Hemoglobin": 0.5,    # g/dL
    "Platelets": 30.0,    # 10^9/L

    # renal function
    "Creatinine": 0.2,    # mg/dL
    "Urea": 10.0,         # mg/dL

    # electrolytes
    "Sodium": 2.0,        # mmol/L
    "Potassium": 0.3,     # mmol/L
}


def classify_trajectory_direction(row):

    delta = row["delta_3_7_vs_0_3"]
    biomarker = row["biomarker"]

    if pd.isna(delta):
        return np.nan

    threshold = trajectory_delta_thresholds.get(biomarker, np.nan)

    if pd.isna(threshold):
        return np.nan

    if delta <= -threshold:
        return "Decreasing"

    elif delta >= threshold:
        return "Increasing"

    else:
        return "Stable"


df_window_compare["trajectory_direction"] = (
    df_window_compare.apply(
        classify_trajectory_direction,
        axis=1
    )
)

print("Trajectory direction distribution:")
print(
    df_window_compare["trajectory_direction"]
    .value_counts(dropna=False)
)

display(df_window_compare.head())

In [ ]:
# ============================================================
# TRAJECTORY DIRECTION THRESHOLDS
# ============================================================

trajectory_thresholds_table = pd.DataFrame(
    [
        {
            "biomarker": biomarker,
            "delta_threshold": threshold,
            "unit": {
                "CRP": "mg/L",
                "WBC": "10^9/L",
                "Neutrophils": "10^9/L",
                "Lymphocytes": "10^9/L",
                "Hemoglobin": "g/dL",
                "Platelets": "10^9/L",
                "Creatinine": "mg/dL",
                "Urea": "mg/dL",
                "Sodium": "mmol/L",
                "Potassium": "mmol/L",
            }.get(biomarker, ""),
            "interpretation": "Absolute change from mean days 0–3 to mean days 3–7"
        }
        for biomarker, threshold in trajectory_delta_thresholds.items()
    ]
)

display(trajectory_thresholds_table)

In [ ]:
# ============================================================
# TRAJECTORY DIRECTION DISTRIBUTION BY BIOMARKER
# ============================================================

trajectory_direction_distribution = (
    df_window_compare
    .groupby(
        [
            "biomarker",
            "trajectory_direction"
        ],
        dropna=False
    )
    .agg(
        n=("episode_key", "nunique")
    )
    .reset_index()
)

trajectory_direction_distribution["pct_within_biomarker"] = (
    trajectory_direction_distribution
    .groupby("biomarker")["n"]
    .transform(lambda x: 100 * x / x.sum())
)

display(trajectory_direction_distribution)

In [ ]:
# ============================================================
# OUTCOMES BY TRAJECTORY DIRECTION
# ============================================================

trajectory_direction_outcomes = (
    df_window_compare
    .dropna(subset=["trajectory_direction"])
    .groupby(
        [
            "biomarker",
            "trajectory_direction"
        ]
    )
    .agg(
        n=("episode_key", "nunique"),
        composite_rate=(
            "composite_inhospital_or_palliative",
            "mean"
        ),
        mortality_90d_rate=(
            "mortality_90d_post_discharge",
            "mean"
        ),
    )
    .reset_index()
)

display(
    trajectory_direction_outcomes
    .sort_values(
        "mortality_90d_rate",
        ascending=False
    )
    .head(50)
)

In [ ]:
# ============================================================
# PLOT: OUTCOME RATES BY TRAJECTORY DIRECTION
# ============================================================

for outcome_col, outcome_label in {
    "composite_rate": "Composite outcome rate",
    "mortality_90d_rate": "90-day mortality rate",
}.items():

    plt.figure(figsize=(12, 5))

    sns.barplot(
        data=trajectory_direction_outcomes,
        x="biomarker",
        y=outcome_col,
        hue="trajectory_direction"
    )

    plt.title(outcome_label + " by biomarker trajectory direction")
    plt.xlabel("Biomarker")
    plt.ylabel(outcome_label)
    plt.xticks(rotation=45)
    plt.legend(title="Trajectory direction")

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# SUMMARY TABLE: MEDIAN BIOMARKER BY OUTCOME
# ============================================================

summary_rows = []

for outcome_col, outcome_label in {
    "composite_inhospital_or_palliative": "Composite outcome",
    "mortality_90d_post_discharge": "90-day mortality",
}.items():

    for biomarker in CORE_BIOMARKERS:

        temp = df_window_compare[
            (df_window_compare["biomarker"] == biomarker)
            & df_window_compare[outcome_col].notna()
        ].copy()

        for feature in [
            "mean_0_3",
            "mean_3_7",
            "delta_3_7_vs_0_3",
        ]:

            for outcome_value in [0, 1]:

                x = temp.loc[
                    temp[outcome_col] == outcome_value,
                    feature
                ].dropna()

                summary_rows.append({
                    "outcome": outcome_label,
                    "biomarker": biomarker,
                    "feature": feature,
                    "outcome_value": outcome_value,
                    "n": len(x),
                    "median": x.median() if len(x) > 0 else np.nan,
                    "q1": x.quantile(0.25) if len(x) > 0 else np.nan,
                    "q3": x.quantile(0.75) if len(x) > 0 else np.nan,
                })

summary_biomarker_windows = pd.DataFrame(summary_rows)

display(summary_biomarker_windows.head(30))

In [ ]:
# ============================================================
# QC: CHECK OLD DELTA COLUMN NAMES
# ============================================================

old_delta_cols = [
    c for c in df_window_compare.columns
    if "delta_mean_3_7_minus_0_3" in c
    or "delta_median_3_7_minus_0_3" in c
]

print("Old delta columns still present:", old_delta_cols)

assert len(old_delta_cols) == 0, "Old delta column names still present."

In [ ]:
# ============================================================
# SAVE OUTPUTS
# ============================================================

df_traj.to_parquet(
    OUTPUT_05 / "05_daily_labs_with_outcomes.parquet",
    index=False
)

daily_summary.to_excel(
    OUTPUT_05 / "05_daily_median_trajectories.xlsx",
    index=False
)

availability_pivot.to_excel(
    OUTPUT_05 / "05_lab_availability_by_day.xlsx"
)

df_window_compare.to_parquet(
    OUTPUT_05 / "05_window_comparison_dataset.parquet",
    index=False
)

window_comparisons_all.to_excel(
    OUTPUT_05 / "05_window_comparisons_fdr_effect_sizes.xlsx",
    index=False
)

summary_biomarker_windows.to_excel(
    OUTPUT_05 / "05_summary_biomarker_windows.xlsx",
    index=False
)

effect_size_summary.to_excel(
    OUTPUT_05 / "05_effect_size_summary.xlsx",
    index=False
)

trajectory_thresholds_table.to_excel(
    OUTPUT_05 / "05_trajectory_direction_thresholds.xlsx",
    index=False
)

trajectory_direction_distribution.to_excel(
    OUTPUT_05 / "05_trajectory_direction_distribution.xlsx",
    index=False
)

trajectory_direction_outcomes.to_excel(
    OUTPUT_05 / "05_trajectory_direction_outcomes.xlsx",
    index=False
)

print("Outputs saved in:", OUTPUT_05)

In [ ]:
# ============================================================
# FINAL SUMMARY
# ============================================================

print("=" * 80)
print("05 BIOMARKER TRAJECTORIES SUMMARY")
print("=" * 80)

print("\nTrajectory dataset")
print("Rows:", df_traj.shape[0])
print("Episodes:", df_traj["episode_key"].nunique())
print("Patients:", df_traj["pubID"].nunique())
print("Biomarkers:", df_traj["biomarker"].nunique())

print("\nAvailability by day")
display(availability_pivot)

print("\nWindow comparison dataset")
print("Rows:", df_window_compare.shape[0])
print("Episodes:", df_window_compare["episode_key"].nunique())

print("\nNominally significant comparisons p < 0.05")
print(
    window_comparisons_all["p_value"].lt(0.05).sum(),
    "/",
    window_comparisons_all["p_value"].notna().sum()
)

print("\nFDR-significant comparisons")
print(
    window_comparisons_all["significant_fdr"].sum(),
    "/",
    window_comparisons_all["significant_fdr"].notna().sum()
)

print("\nEffect size magnitude summary")
display(effect_size_summary)

print("\nTop FDR-significant comparisons")
display(
    window_comparisons_all[
        window_comparisons_all["significant_fdr"]
    ]
    .sort_values("p_value_fdr")
    .head(20)
)

print("\nTrajectory direction thresholds")
display(trajectory_thresholds_table)

print("\nTrajectory direction distribution")
display(trajectory_direction_distribution)

print("\nTop trajectory-outcome direction patterns")
display(
    trajectory_direction_outcomes
    .sort_values(
        "mortality_90d_rate",
        ascending=False
    )
    .head(20)
)

print("""
Interpretation note:
This notebook evaluates longitudinal laboratory trajectories using both
absolute and normalized biomarker values.

LOWESS curves are used only for visual interpretation. Later hospitalization
days have fewer contributing patients due to discharge, death, and variable
monitoring intensity; therefore, small late-window oscillations should be
interpreted cautiously.

Statistical comparisons include both effect sizes and FDR correction to
distinguish clinically meaningful differences from purely statistically
significant findings.

Trajectory direction categories are descriptive and biomarker-specific.
They classify whether mean values from days 3–7 increased, decreased, or
remained stable compared with days 0–3.

These labels describe the direction of biomarker change only and do not imply
clinical improvement or deterioration per se. For example, decreasing
inflammatory markers may reflect recovery, whereas decreasing sodium levels
may indicate clinical worsening. Similarly, increasing lymphocyte counts may
reflect immune recovery, whereas increasing neutrophils or WBC may reflect
persistent or worsening inflammatory activation.
""")